# Experiment 009: 1.5B SFT-to-GRPO run, seed 2 (Colab)

One run: SFT-to-GRPO seed 2 under the unchanged frozen protocol, identical to
seeds 0 and 1 except the seed (pair-tested across all three configs). It is
the last cell of three-seeds-everywhere grid: after it, every scale's
GRPO spread rests on three fully recorded draws. Base evals reused from
the shared 009 directory.

Recorded environment must match the seed-1 sibling exactly; the deps cell
asserts it and stops the session otherwise. ~5h on an A100-80GB, one
session likely, resume-safe throughout.

If the run OOMs: do NOT rerun as-is. Archive the failed directory into
failed_attempts following the registered OOM procedure, then restart from
step zero.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 494.5       # fill from the resources panel
unit_rate = 6.77       # fill from the resources panel
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '009_colab_1p5b_seed2', 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260802_201926 | cu_before = 494.5
Mounted at /content/drive
saved open session: 20260802_201926
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [2]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 1.5b run, kept for uniformity per the recorded decision'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-9cde9eb3-9346-7384-cc46-b3c0bdf75e8b)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3198, done.
remote: Counting objects: 100% (682/682), done.
remote: Compressing objects: 100% (452/452), done.
remote: Total 3198 (delta 428), reused 479 (delta 230), pack-reused 2516 (from 1)
Receiving objects: 100% (3198/3198), 91.77 MiB | 31.50 MiB/s, done.
Resolving deltas: 100% (2047/2047), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')

# match the recorded 003 seed-1 environment before training
# require the reference manifest so a missing record stops the session
import importlib, json, os
ref = 'results/009_COLAB_FULL_B15_sft_grpo_seed1/manifest.json'
assert os.path.exists(ref), 'reference manifest missing, pull the repo with results before training'
par = json.load(open(ref))['versions']
for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
    have = importlib.import_module(lib).__version__
    assert have == par[lib], f'{lib} {have} differs from the recorded {par[lib]}'
print('environment matches the recorded 009 seed-1 versions')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.7 MB/s eta 0:00:00
deps installed
environment matches the recorded 009 seed-1 versions


## 4. W&B key (automatic)

Reminder: colab secrets are granted per notebook, switch on access for
`WANDB_API_KEY` under the key icon before running. This key is only used for experiment tracking.


In [5]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record and free space

In [6]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 20, 'need >=15GB free on drive for the 1.5b run'

open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260802_201926.json
drive free: 179 GB


## 6. Build the data splits

In [7]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 16.9MB/s]

main/train-00000-of-00001.parquet: downloading bytes:  96% 2.21M/2.31M [00:00<00:00, 3.04MB/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 3.13MB/s,  227kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 3.13MB/s,  228kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   8% 32.8k/419k [00:00<00:07, 50.5kB/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 598kB/s, 41.3kB/s  ] 
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 598kB/s, 41.3kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 397777.02 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 251079.56 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 84.3MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 151MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00

## 7. Preflight: the shared 009 base evals must exist

In [8]:
import os
D = '/content/drive/MyDrive/inm363'
be = os.path.join(D, 'results', 'base_evals', '009_COLAB_FULL_B15')
for f in ('base_eval.json', 'base_logprob.json'):
    assert os.path.exists(os.path.join(be, f)), f'shared 009 base eval missing: {f}'
print('shared base evals present, the run will skip them')

shared base evals present, the run will skip them


## 8. Wiring check

In [9]:
!python scripts/run.py --config configs/009_colab_full_b15_seed2.yaml

env=COLAB mode=FULL scale=B15 condition=SFT_GRPO
model=Qwen/Qwen2.5-1.5B device=cuda stages=['base_eval', 'sft', 'sft_eval', 'grpo', 'grpo_eval']
steps=500 N=8 seed=2
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/009_COLAB_FULL_B15_sft_grpo_seed2/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/009_COLAB_FULL_B15_sft_grpo_seed2/
local smoke wiring ok


## 9. The run

**If colab disconnects:** re-run this cell, finished stages skip.

In [ ]:
# both cost fields must be filled before the long run starts
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
# python loop with ! calls so the logs stream live, exit code checked so a
# failed run stops the session instead of hiding behind the next one
for cfg in ('009_colab_full_b15_seed2',):
    print(f'===== {cfg} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{cfg}.yaml --stage all
    if _exit_code != 0:
        raise SystemExit(f'{cfg} failed with exit code {_exit_code}, see the OOM protocol before retrying')

Streaming output truncated to the last 5000 lines.
│ │ B. 2.25       │ square        │               │              │           │ │
│ │ C. 6.6        │ units.</thin… │               │              │           │ │
│ │ D. 3.3        │ <answer>B</a… │               │              │           │ │
│ │ <think>       │               │               │              │           │ │
│ ├───────────────┼───────────────┼───────────────┼──────────────┼───────────┤ │
│ │ Answer the    │ Since $y \ge  │          0.00 │         0.20 │     -0.93 │ │
│ │ multiple      │ |x|$ and $y   │               │              │           │ │
│ │ choice        │ \le -|x|+3$,  │               │              │           │ │
│ │ question.     │ its common    │               │              │           │ │
│ │ Think through │ region is     │               │              │           │ │
│ │ it step by    │ shown in the  │               │              │           │ │
│ │ step inside   │ figure below. │               │       

## 10. Peek at the results

In [ ]:
import json, os
D = '/content/drive/MyDrive/inm363'
for tag in ('009_COLAB_FULL_B15_sft_grpo_seed0', '009_COLAB_FULL_B15_sft_grpo_seed1',
            '009_COLAB_FULL_B15_sft_grpo_seed2'):
    sm = os.path.join(D, 'results', tag, 'summary.json')
    if os.path.exists(sm):
        j = json.load(open(sm))
        print(f"{tag}: sft {j['sft_acc']}  grpo {j['grpo_acc']}")
    else:
        print(f'{tag}: no summary yet')

009_COLAB_FULL_B15_sft_grpo_seed0: sft {'easy': 0.6382252559726962, 'medium': 0.5763546798029556, 'hard': 0.30303030303030304}  grpo {'easy': 0.8088737201365188, 'medium': 0.6272577996715928, 'hard': 0.44696969696969696}
009_COLAB_FULL_B15_sft_grpo_seed1: sft {'easy': 0.6279863481228669, 'medium': 0.5320197044334976, 'hard': 0.29545454545454547}  grpo {'easy': 0.7679180887372014, 'medium': 0.5977011494252874, 'hard': 0.39646464646464646}
009_COLAB_FULL_B15_sft_grpo_seed2: sft {'easy': 0.602957906712173, 'medium': 0.5073891625615764, 'hard': 0.2601010101010101}  grpo {'easy': 0.7906712172923777, 'medium': 0.632183908045977, 'hard': 0.42424242424242425}


## 11. Close the units record

In [ ]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 448.12     # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
